In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import json
import pickle

# --- DATA PREPARATION & ANTI-LEAKAGE FILTERING ---
# Load datasets
file_path = '/content/drive/MyDrive/capstone project/'
customers = pd.read_csv(file_path + "customers.csv")
orders = pd.read_csv(file_path + "orders.csv")
tickets = pd.read_csv(file_path + "support_tickets.csv")
web_events = pd.read_csv(file_path + "web_events_snapshot.csv")
labels = pd.read_csv(file_path + "churn_labels.csv")

# Enforce strict snapshot date constraint (Replace with data dictionary date)
SNAPSHOT_DATE = pd.to_datetime('2026-01-01')

orders['order_date'] = pd.to_datetime(orders['order_date'])
tickets['ticket_date'] = pd.to_datetime(tickets['ticket_date'])
web_events['snapshot_date'] = pd.to_datetime(web_events['snapshot_date'])

# Filter out any post-snapshot activity
valid_orders = orders[orders['order_date'] <= SNAPSHOT_DATE]
valid_tickets = tickets[tickets['ticket_date'] <= SNAPSHOT_DATE]
valid_events = web_events[web_events['snapshot_date'] <= SNAPSHOT_DATE]

# --- FEATURE EXTRACTION ---
# 1. Order Features
order_feats = valid_orders.groupby('customer_id').agg(
    total_spend=('gross_amount', 'sum'),
    order_count=('order_id', 'nunique')
).reset_index()

# 2. Support Ticket Features
ticket_feats = valid_tickets.groupby('customer_id').size().reset_index(name='ticket_count')

# 3. Web Event Features
event_feats = valid_events.groupby('customer_id').size().reset_index(name='web_sessions')

# Master Feature Matrix Assembly
feature_matrix = customers[['customer_id']].copy()
feature_matrix = feature_matrix.merge(order_feats, on='customer_id', how='left')
feature_matrix = feature_matrix.merge(ticket_feats, on='customer_id', how='left')
feature_matrix = feature_matrix.merge(event_feats, on='customer_id', how='left')
feature_matrix = feature_matrix.fillna(0)

# Merge with Ground Truth Labels
dataset = feature_matrix.merge(labels, on='customer_id', how='inner')

# --- DATA SPLIT ---
X = dataset.drop(columns=['customer_id', 'churn_next_60d', 'snapshot_date', 'split'])
y = dataset['churn_next_60d']

# Split: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Split Completed Successully:")
print(f"Train size: {X_train.shape[0]} | Val size: {X_val.shape[0]} | Test size: {X_test.shape[0]}")

Dataset Split Completed Successully:
Train size: 1680 | Val size: 360 | Test size: 360


In [8]:
# --- MODEL 1: SIMPLE BASELINE (LOGISTIC REGRESSION) ---
baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(X_train_scaled, y_train)
baseline_preds = baseline_model.predict_proba(X_val_scaled)[:, 1]

# --- MODEL 2: STRONG ENSEMBLE (XGBOOST CLASSIFIER) ---
strong_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train), # Handles class imbalance
    random_state=42,
    eval_metric='logloss'
)
strong_model.fit(X_train, y_train) # Tree models don't strictly require scaling
strong_preds = strong_model.predict_proba(X_val)[:, 1]

print("✅ Both models successfully trained.")

✅ Both models successfully trained.


In [9]:
# Select business-justified threshold (e.g., 0.35 to catch high-risk customers proactively)
BUSINESS_THRESHOLD = 0.35

y_val_pred_labels = (strong_preds >= BUSINESS_THRESHOLD).astype(int)

# Compile validation metrics
metrics_dict = {
    "baseline_val_roc_auc": float(roc_auc_score(y_val, baseline_preds)),
    "strong_model_val_roc_auc": float(roc_auc_score(y_val, strong_preds)),
    "chosen_threshold": BUSINESS_THRESHOLD,
    "classification_report_val": classification_report(y_val, y_val_pred_labels, output_dict=True)
}

# Export required metrics file
with open("metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=4)

# Export the strong model artifact for Part 4 productionization
with open("model.pkl", "wb") as f:
    pickle.dump({'model': strong_model, 'scaler': scaler}, f)

print("\n=== Validation Performance (Strong Model) ===")
print(classification_report(y_val, y_val_pred_labels))
print(f"Saved artifacts: 'metrics.json' and 'model.pkl' [cite: 83, 84]")


=== Validation Performance (Strong Model) ===
              precision    recall  f1-score   support

           0       0.69      0.48      0.57       191
           1       0.56      0.76      0.65       169

    accuracy                           0.61       360
   macro avg       0.63      0.62      0.61       360
weighted avg       0.63      0.61      0.60       360

Saved artifacts: 'metrics.json' and 'model.pkl' [cite: 83, 84]
